### Inference 3D

In [ ]:
import os
import subprocess
import shutil
import gc
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import tifffile
import mrcfile
import psutil

# -----------------------------------------------------------------------------
# 1) SET nnU-NET ENVIRONMENT VARIABLES (v2 names!)
# -----------------------------------------------------------------------------
os.environ['nnUNet_raw']          = '/media/home/DATA10TB/MITOCHONDRIA/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/media/home/DATA10TB/MITOCHONDRIA/nnUnet_preprocessed'
os.environ['nnUNet_results']      = '/media/home/DATA10TB/MITOCHONDRIA/nnUNet_results'

# -----------------------------------------------------------------------------
# 2) HELPERS
# -----------------------------------------------------------------------------
def print_memory_usage(msg):
    proc = psutil.Process()
    print(f"{msg} – memory: {proc.memory_info().rss/(1024**3):.2f} GB")

def ensure_dir(d):
    os.makedirs(d, exist_ok=True)

def clear_dir(d):
    if os.path.exists(d):
        for fn in os.listdir(d):
            fp = os.path.join(d, fn)
            if os.path.isfile(fp) or os.path.islink(fp):
                os.remove(fp)
            else:
                shutil.rmtree(fp)

def load_stack(fp):
    """Load a 3D stack; take channel 1 if TIFF has >1 channel, fix signed-8."""
    if fp.lower().endswith('.mrc'):
        with mrcfile.open(fp,'r') as m:
            arr = m.data
    else:
        with tifffile.TiffFile(fp) as tif:
            arr = tif.asarray()
            if arr.ndim == 4:
                arr = arr[...,1] if arr.shape[-1] > 1 else arr[...,0]
    if np.issubdtype(arr.dtype, np.integer) and arr.dtype.itemsize == 1 and np.any(arr < 0):
        arr = (arr.astype(np.int16) + 256).astype(np.uint8)
    return arr

def display_slice(slice_2d):
    """Show one 2D slice plus its histogram side by side."""
    slice_2d = slice_2d.astype(np.uint8)
    fig, (ax1, ax2) = plt.subplots(1,2, figsize=(10,5))
    ax1.imshow(slice_2d, cmap='gray')
    ax1.axis('off'); ax1.set_title('Middle slice')
    ax2.hist(slice_2d.ravel(), bins=256)
    ax2.set_title('Histogram')
    plt.tight_layout(); plt.show()

# -----------------------------------------------------------------------------
# 3) 3D FULL-VOLUME INFERENCE (uses nnUNetv2_predict)
# -----------------------------------------------------------------------------
def run_3d_full_volume_inference(stack, tmp_input_dir, tmp_output_dir,
                                 d_id, fold, trainer,
                                 save_probs=True,
                                 step_size=0.5):
    """
    Run nnU-Net 3D fullres inference on the entire volume.

    Args:
        stack: 3D numpy array (Z,Y,X)
        tmp_input_dir: temporary directory for input .nii.gz file
        tmp_output_dir: temporary directory for nnUNet output
        d_id, fold, trainer: nnUNet dataset id, fold, trainer name
        save_probs: whether to save soft probabilities
        step_size: fraction of patch size to slide (0–1)
    Returns:
        prob_map: 3D numpy array of class-1 probabilities (float32)
    """
    # prepare temporary input
    ensure_dir(tmp_input_dir)
    clear_dir(tmp_input_dir)
    nib.save(
        nib.Nifti1Image(stack.astype(np.float32), np.eye(4)),
        os.path.join(tmp_input_dir, 'case_0000.nii.gz')
    )

    # prepare temporary output
    ensure_dir(tmp_output_dir)
    clear_dir(tmp_output_dir)

    # run nnU-Net prediction
    cmd = [
        'nnUNetv2_predict',
        '-i', tmp_input_dir,
        '-o', tmp_output_dir,
        '-d', str(d_id),
        '-f', str(fold),
        '-tr', trainer,
        '-c', '3d_fullres',
        '-step_size', str(step_size),
    ]
    if save_probs:
        cmd.append('--save_probabilities')
    
    try:
        # redirect both stdout and stderr to DEVNULL to silence everything
        subprocess.run(cmd,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL,
                       check=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"nnUNetv2_predict (3D) failed (exit code {e.returncode})")

    # find and load the .npz file that nnU-Net produced
    npz_files = [f for f in os.listdir(tmp_output_dir) if f.endswith('.npz')]
    if not npz_files:
        raise FileNotFoundError(f"No .npz found in {tmp_output_dir}")
    if len(npz_files) > 1:
        print("Warning: multiple .npz files found, loading the first one:", npz_files)
    prob_path = os.path.join(tmp_output_dir, npz_files[0])

    data = np.load(prob_path)
    arr = data[data.files[0]]  # either shape (C,Z,Y,X) or (Z,Y,X)
    if arr.ndim == 4:
        # class-1 probability channel
        return arr[1]
    else:
        # if only one channel, assume it's already the prob map
        return arr

# -----------------------------------------------------------------------------
# 4) PROCESS ONE FILE AND SAVE OVERLAY
# -----------------------------------------------------------------------------
def process_file_3d(fp, tmp_input_dir, tmp_output_dir,
                    d_id, fold, trainer,
                    save_probs=True, step_size=0.5):
    """Process one file and save overlay to the same directory as input."""
    stack = load_stack(fp)
    
    prob = run_3d_full_volume_inference(
        stack, tmp_input_dir, tmp_output_dir, d_id, fold, trainer,
        save_probs=save_probs, step_size=step_size
    )

    # build RGB overlay: green=orig, red=seg
    rgb = np.zeros((*stack.shape,3), dtype=np.uint8)
    rgb[...,1] = stack
    mask = (prob * 255).clip(0,255).astype(np.uint8)
    if mask.shape != stack.shape:
        mask = mask.transpose((2,1,0))
    rgb[...,0] = mask

    # Save overlay in the same directory as the input file
    p = Path(fp)
    ov_fp = p.parent / f"{p.stem}_3D_OVERLAY_model{d_id}_fold{fold}.tiff"
    tifffile.imwrite(str(ov_fp), rgb, photometric='rgb')
    print(f"→ Saved overlay: {ov_fp}")

# -----------------------------------------------------------------------------
# 5) MAIN LOOP
# -----------------------------------------------------------------------------
def traverse(top_dir, tmp_input_dir, tmp_output_dir,
             d_id, fold, trainer,
             save_probs=True, step_size=0.5):
    # scan and print each file
    print("Scanning for .mrc/.tif files…")
    files = []
    for root, _, fs in os.walk(top_dir):
        for f in fs:
            if f.lower().endswith(('.mrc', '.tif', '.tiff')):
                fp = os.path.join(root, f)
                files.append(fp)
                print(f"Found: {fp}")

    # summary and prompt
    total = len(files)
    print(f"Total files to process: {total}")
    #if input("Continue? (yes/no): ").strip().lower() != 'yes':
    #    return

    # process with numbered progress
    for i, fp in enumerate(files, 1):
        print(f"\n=== [{i}/{total}] {fp} ===")
        stack = load_stack(fp)
        display_slice(stack[stack.shape[0]//2])
        process_file_3d(
            fp, tmp_input_dir, tmp_output_dir, d_id, fold, trainer,
            save_probs=save_probs, step_size=step_size
        )
        gc.collect()
        if i % 10 == 0 or i == total:
            print_memory_usage(f"Processed {i}/{total}")

    print_memory_usage("All done")

### 3D MitoEye

In [ ]:
top_directory     = '/media/home/DATA10TB/FULL TEST VOLUME/0'
tmp_input_dir     = './nnunet_input_3d'      # Temporary directory for input
tmp_output_dir    = './nnunet_output_3d'     # Temporary directory for predictions
dataset_id        = 4005
fold              = 0
trainer           = 'nnUNetTrainer_100epochs'
save_probabilities= True
step_size         = 0.5

traverse(
    top_directory, tmp_input_dir, tmp_output_dir,
    dataset_id, fold, trainer,
    save_probs=save_probabilities,
    step_size=step_size
)

In [ ]:
top_directory     = '/media/home/DATA10TB/FULL TEST VOLUME/1'
tmp_input_dir     = './nnunet_input_3d'      # Temporary directory for input
tmp_output_dir    = './nnunet_output_3d'     # Temporary directory for predictions
dataset_id        = 4005
fold              = 1
trainer           = 'nnUNetTrainer_100epochs'
save_probabilities= True
step_size         = 0.5

traverse(
    top_directory, tmp_input_dir, tmp_output_dir,
    dataset_id, fold, trainer,
    save_probs=save_probabilities,
    step_size=step_size
)

In [ ]:
top_directory     = '/media/home/DATA10TB/FULL TEST VOLUME/2'
tmp_input_dir     = './nnunet_input_3d'      # Temporary directory for input
tmp_output_dir    = './nnunet_output_3d'     # Temporary directory for predictions
dataset_id        = 4005
fold              = 2
trainer           = 'nnUNetTrainer_100epochs'
save_probabilities= True
step_size         = 0.5

traverse(
    top_directory, tmp_input_dir, tmp_output_dir,
    dataset_id, fold, trainer,
    save_probs=save_probabilities,
    step_size=step_size
)

In [ ]:
top_directory     = '/media/home/DATA10TB/FULL TEST VOLUME/3'
tmp_input_dir     = './nnunet_input_3d'      # Temporary directory for input
tmp_output_dir    = './nnunet_output_3d'     # Temporary directory for predictions
dataset_id        = 4005
fold              = 3
trainer           = 'nnUNetTrainer_100epochs'
save_probabilities= True
step_size         = 0.5

traverse(
    top_directory, tmp_input_dir, tmp_output_dir,
    dataset_id, fold, trainer,
    save_probs=save_probabilities,
    step_size=step_size
)

In [ ]:
top_directory     = '/media/home/DATA10TB/FULL TEST VOLUME/4'
tmp_input_dir     = './nnunet_input_3d'      # Temporary directory for input
tmp_output_dir    = './nnunet_output_3d'     # Temporary directory for predictions
dataset_id        = 4005
fold              = 4
trainer           = 'nnUNetTrainer_100epochs'
save_probabilities= True
step_size         = 0.5

traverse(
    top_directory, tmp_input_dir, tmp_output_dir,
    dataset_id, fold, trainer,
    save_probs=save_probabilities,
    step_size=step_size
)

### 3D VESSEL12

In [ ]:
top_directory     = '/media/home/DATA10TB/VESSEL12/RAW2'
chunk_directory   = './chunks'
output_directory  = './predictions'
dataset_id        = 3005
fold              = 2
trainer           = 'nnUNetTrainer_100epochs'
save_probabilities= True
chunk_size        = [128,352,352]
stride            = [64,176,176]

traverse(
  top_directory, chunk_directory, output_directory,
  dataset_id, fold, trainer, save_probabilities,
  chunk_size, stride
)

In [ ]:
top_directory     = '/media/home/DATA10TB/VESSEL12/RAW2'
tmp_input_dir     = './nnunet_input_3d'      # Temporary directory for input
tmp_output_dir    = './nnunet_output_3d'     # Temporary directory for predictions
dataset_id        = 3005
fold              = 2
trainer           = 'nnUNetTrainer_100epochs'
save_probabilities= True
step_size         = 0.5

traverse(
    top_directory, tmp_input_dir, tmp_output_dir,
    dataset_id, fold, trainer,
    save_probs=save_probabilities,
    step_size=step_size
)